In [1]:
import duckdb
import random
from functions.analyte import ANALYTES
from networks.cnn_dilated_convolutions import CNNModel
from networks.auto_encoder import AutoencoderModel
from functions.evaluation import evaluate

con = duckdb.connect('../capillary.db')

df = con.execute(""" 
                 SELECT row_id, age, gender, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data 
                 WHERE value IS NOT NULL
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 AND observation_nr = 1
                 """).df()


df = df[
    (df['fractions'].apply(len) == 6) &
    (df['boundaries'].apply(len) == 12)
]

protein_cols = [a.col for a in ANALYTES[:8]]  # bara de 8 första
df = df.dropna(subset=protein_cols)
y = df['label']

cnn = CNNModel()
ae = AutoencoderModel()
df = cnn.predict(df)
df = ae.predict(df)
con.close()


In [5]:
import random

con = duckdb.connect('../data_processing/application.db')
a = con.execute("SHOW TABLES").fetchall()
print(a)
ids = set(df.loc[(df['label'] == 0) & (df['cnn_probability'] > 0.15), 'id'])
ids |= set(df.loc[(df['label'] == 1) & (df['cnn_probability'] <= 0.2), 'id'])
ids |= set(df.loc[(df['proportion_gamma_region'] > 70) & (df['cnn_probability'] <= 0.2), 'id'])
ids |= set(df.loc[(df['proportion_gamma_region'] < 70) & (df['cnn_probability'] > 0.2), 'id'])

random_ids = set(random.sample(list(df['id']), 1000))
ids |= random_ids
print(len(ids))
con.executemany("INSERT OR IGNORE INTO difficult_cases VALUES (?)", [(id,) for id in ids])


ids = set(df.loc[(df['label'] == 0) & (df['cnn_probability'] > 0.8), 'id'])
ids |= set(df.loc[(df['label'] == 1) & (df['cnn_probability'] <= 0.15), 'id'])
ids |= set(df.loc[(df['proportion_gamma_region'] > 62) & (df['cnn_probability'] <= 0.2), 'id'])
random_ids = set(random.sample(list(df['id']), 100))
ids |= random_ids
print(len(ids))
con.executemany("INSERT OR IGNORE INTO most_difficult_cases VALUES (?)",[(id,) for id in ids])
con.close()


[('classifications',), ('difficult_cases',), ('difficult_curves',), ('most_difficult_cases',), ('users',)]
7158
804
